In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer



In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')


In [4]:
df.isnull().sum()

df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')


numerical_columns = list(df.dtypes[df.dtypes != 'str'].index)
numerical_columns.remove('converted')

df[categorical_columns] = df[categorical_columns].fillna('NA')
df[numerical_columns] = df[numerical_columns].fillna(0.0)



In [5]:
numerical_columns


['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [6]:
df.isnull().sum()


lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [7]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [8]:
numerical_columns = ['number_of_courses_viewed', 'annual_income','interaction_count', 'lead_score']

corr = df[numerical_columns].corr()
corr


,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.161300,0.721609,0.757204
annual_income,0.161300,1.000000,0.122842,0.229496
interaction_count,0.721609,0.122842,1.000000,0.915746
lead_score,0.757204,0.229496,0.915746,1.000000


In [9]:


interaction_count_and_lead_score = corr.loc['interaction_count', 'lead_score']
number_of_courses_viewed_and_lead_score = corr.loc['number_of_courses_viewed', 'lead_score']
number_of_courses_viewed_and_interaction_count = corr.loc['number_of_courses_viewed', 'interaction_count']
annual_income_and_interaction_count = corr .loc['annual_income', 'interaction_count']

print(f"Correlation between interaction_count and lead_score: {interaction_count_and_lead_score}")
print(f"Correlation between number_of_courses_viewed and lead_score: {number_of_courses_viewed_and_lead_score}")
print(f"Correlation between number_of_courses_viewed and interaction_count: {number_of_courses_viewed_and_interaction_count}")
print(f"Correlation between annual_income and interaction_count: {annual_income_and_interaction_count}")

Correlation between interaction_count and lead_score: 0.9157457980418697
Correlation between number_of_courses_viewed and lead_score: 0.7572042392104886
Correlation between number_of_courses_viewed and interaction_count: 0.7216090038937035
Correlation between annual_income and interaction_count: 0.12284235135955959


In [10]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']


In [11]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37
1,events,finance,employed,south_america,71662.0,0,0,0.09
2,referral,NA,employed,asia,40220.0,2,5,0.55
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45
4,events,technology,employed,north_america,89316.0,1,5,0.00


In [12]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical_columns].apply(mutual_info_converted_score)
mi.sort_values(ascending=False).round(2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [33]:
features = categorical_columns + numerical_columns
dv = DictVectorizer(sparse=False)

train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = float((y_pred == y_val).mean())

print(f"Accuracy: {accuracy:.2f}")  

Accuracy: 0.65


In [41]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)

    train_dicts = df_train[features].to_dict(orient='records')
    val_dicts = df_val[features].to_dict(orient='records')

    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    accuracy = float((y_pred == y_val).mean())
    
    return accuracy

base_accuracy = train_and_score(features)

scores = pd.DataFrame(columns=['eliminated_feature', 'accuracy', 'difference'])

for feature in features:
    subset = features.copy()
    subset.remove(feature)

    score = train_and_score(subset)

    scores.loc[len(scores)] = [feature, score, base_accuracy - score]


scores.sort_values(by='difference', ascending=False, inplace=True)
scores 
    



,eliminated_feature,accuracy,difference
6,interaction_count,0.601,0.044
0,lead_source,0.642,0.003
4,number_of_courses_viewed,0.643,0.002
7,lead_score,0.644,0.001
3,location,0.645,0.000
2,employment_status,0.645,0.000
1,industry,0.645,0.000
5,annual_income,0.724,-0.079


In [45]:
diff_C = {0.000001, 0.00001, 0.0001, 0.001}

scores = {}

for c in diff_C:
    accuracy = train_and_score(features, C=c)
    scores[c] = round(accuracy, 3)
    print(f"C={c:.6f}, Accuracy={accuracy:.3f}")



C=0.000010, Accuracy=0.598
C=0.000100, Accuracy=0.613
C=0.000001, Accuracy=0.598
C=0.001000, Accuracy=0.645
